> **DRAFT. The submission form for this project is not set up yet. Do not hand this notebook out.**

# CML3112 Machine Learning: Engineering ML project
**Part A: coded notebook (30 marks) and Part B: decision memo form (15 marks). Together 45 marks, scaled to 40 percent of the course. One sitting of 3 hours, individual work, marked automatically.**
**Sitting:** Tuesday 1 December 2026, 11:00 to 14:00 Kampala time (3 hours; proposed, lecturer to confirm). Your own 3 hours start when you press Start on the entry page and cannot be paused; the class closing time is the last cutoff.

This project uses only what you learned in Weeks 1 to 11. You work on `engineering_data.csv` (building cooling load, in **kW**), and answer two questions: **Question 1** how large will the building cooling load be on equipment the model has never met, and **Question 2** which rows should raise a warning. Group E6 is the test set and is opened **once**.

## How to work
1. Fill in the identity cell below (in the issued notebook it is filled in for you; check it) and run it. Then run the supplied cell **Prepare the data and the libraries**. It creates your CSV, so you need no upload.
2. Work from top to bottom. Replace each `TODO` and each `None`. Keep the function names, the argument order and the variable names exactly as printed. Run each cell and read what it prints.
3. Run the **self-check** cells to see which functions still need work. They are not marked.
4. Part B is a form (the `MEMO` cell). Nothing is written as an essay.
5. Choose **Runtime, then Restart session and run all**. Fix any error, then save with **File, then Save**.
6. Run the last cell, click **Submit test**, open the form link, check your details and press **Submit** there. Submission is finished only when the form says your response has been recorded.

## Time plan (180 minutes)
| Step | Time |
|---|---|
| Setup: sign in, open your notebook, run the supplied cells | 10 min |
| Tasks 1 to 3: cleaning, the split, the three models | 45 min |
| Tasks 4 to 6: warnings, cross-validation, ablation | 45 min |
| Tasks 7 and 8: choose a model, open the test set once | 15 min |
| Task 9: the Part B memo form | 25 min |
| Check, restart and run all, submit | 25 min |
| Spare time (breaks, reconnecting) | 15 min |

## How it is marked
Everything is marked by the lecturer's computer. No person reads your notebook. Your functions are run on **other data that you cannot see** (hidden checks) and your results on your own data file are compared with the correct values. Of the 30 Part A marks, 24 are for the functions and your results, and 6 are for how you worked: the test set opened once and only after you chose, no data leakage (fit preprocessing on the training set only), the same answer every time (keep `random_state=17`), and readable charts. Part B gives 15 marks for the memo form. A list of notebooks that look unusual is checked by the lecturer afterwards.

**Rules (the same rules as Practical Test 1).** Individual work. **Permitted resources:** your lecture slides, your lab notebooks from Classes 1 to 4, the revision guide, the cheatsheet and your own course notes. Write the code yourself.
**No AI or outside help:** do not use Gemini, ChatGPT, Copilot or any other AI tool to generate, complete, rewrite, explain or debug your answers. Do not get answers from another person, collaborate or copy someone else's work.
**Understanding check:** you may be asked to explain your submitted code, or make a small change to it, without AI or another person's help.
**Declaration:** by submitting, I confirm that I completed this project myself without AI assistance or another person's help, using only the permitted resources. I can explain and change the code I submitted.


In [ ]:
FULL_NAME = ""
EMAIL = ""
STUDENT_ID = ""   # your Student Registration Number exactly as on your university record
COHORT = "CML3112"  # Do not edit: identifies your course for marking.
TEST_ID = "ML_PROJECT"  # Do not edit: identifies this assessment.
VARIANT = "C"  # Do not edit.


## Your data
Each row is one synthetic (invented for teaching) observation of **building cooling load**.

| Column | Meaning |
|---|---|
| record_id | Row ID. A repeated ID means a duplicate row. record_id is never a time. |
| equipment_id | Which equipment: E1 to E6. Repeats across rows. E6 is the test set. |
| load_pct | Load in percent (a feature). Valid range 0 to 100, both included. |
| ambient_c | Ambient temperature in degrees Celsius (a feature). May be a missing value. |
| target | The target variable: building cooling load, in **kW**. May be a missing value. |

A **record** is one row as a dictionary. A **list of records** is several rows. Your functions take and return these. Missing values are `None`. Calculate from the lists you are given. Do not use the global tables inside a function, because the marked checks give your functions other data.

In [ ]:
#@title Prepare the data and the libraries - Do not edit this cell
# IUEA-SETUP-CELL  (supplied: do not edit)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

TARGET_NAME = 'building cooling load'
TARGET_UNIT = 'kW'
LIMIT = 54.0          # a row is a WARNING (1) when its target is at or above this limit, in kW
MAX_MAE = 2.0          # the client accepts a typical prediction error (MAE) of at most this many kW (invented teaching number)
MAX_MS = 50                 # and a prediction time of at most this many milliseconds (invented teaching number)
LATENCY_MS = {'baseline': 1, 'linear': 3, 'tree': 30}   # invented teaching numbers
MISSED_COST, FALSE_ALARM_COST = 5, 1   # invented teaching costs (a false negative costs 5, a false positive costs 1)

_CSV_TEXT = """record_id,equipment_id,load_pct,ambient_c,target
1,E1,22.37,34.47,38.407
2,E2,91.01,22.4,60.995
3,E3,58.72,24.77,45.198
4,E4,82.08,22.36,
5,E5,60.79,29.66,50.787
6,E6,49.33,26.38,43.104
7,E1,89.31,31.98,65.515
8,E2,90.41,21.9,57.699
9,E3,-5,20.67,35.882
10,E4,72.64,32.06,57.674
11,E5,72.68,29.1,55.97
12,E6,94.29,30.45,68.556
13,E1,94.77,33.02,69.907
14,E2,80.89,,59.38
15,E3,29.57,32.23,38.487
16,E4,87.45,26.39,61.269
17,E5,40.28,27.41,41.007
18,E6,94.54,33.43,68.342
19,E1,80.36,26.24,57.822
20,E2,22.88,34.1,39.059
21,E3,67.94,22.01,48.741
22,E4,25.42,25.33,32.664
23,E5,28.0,20.29,28.823
24,E6,22.02,32.51,36.935
25,E1,67.27,22.39,48.128
26,E2,39.0,26.57,38.56
27,E3,84.07,29.48,63.129
28,E4,31.55,21.14,31.547
29,E5,63.34,19.2,43.454
30,E6,53.79,23.94,43.614
31,E1,51.57,23.78,41.214
32,E2,51.52,26.99,44.905
33,E3,42.25,24.76,39.504
34,E4,57.81,20.71,42.946
35,E5,43.8,27.46,40.693
36,E6,39.19,31.94,43.274
37,E1,31.17,33.63,40.632
38,E2,45.19,34.09,48.076
39,E3,72.04,34.04,60.431
40,E4,80.22,18.75,51.055
41,E5,80.96,22.35,53.786
42,E6,58.15,27.38,48.972
43,E1,44.02,24.68,39.695
44,E2,21.55,28.94,32.771
45,E3,26.18,22.36,29.647
46,E4,49.25,34.55,49.962
47,E5,84.52,18.71,54.509
48,E6,84.41,21.02,54.696
49,E1,43.81,20.73,36.212
50,E2,40.74,34.31,44.671
51,E3,74.55,30.14,56.435
52,E4,61.2,31.24,53.169
53,E5,88.56,19.2,55.703
54,E6,89.17,23.59,59.15
55,E1,48.3,27.45,42.965
56,E2,81.4,25.88,57.106
57,E3,77.41,20.31,51.082
58,E4,58.08,20.29,41.585
59,E5,85.25,20.43,54.78
60,E6,20.84,34.27,35.947
61,E1,76.14,19.72,50.753
62,E2,54.04,27.67,47.932
63,E3,51.44,18.09,38.114
64,E4,29.52,24.89,33.449
65,E5,49.36,30.97,48.008
66,E6,38.75,28.65,40.552
67,E1,45.59,19.26,35.875
68,E2,81.23,23.52,55.11
69,E3,28.5,28.05,34.282
70,E4,36.15,19.66,30.143
71,E5,89.35,22.59,57.639
72,E6,93.64,28.1,63.981
73,E1,80.85,21.16,53.536
74,E2,77.24,28.82,57.042
75,E3,30.36,24.85,33.199
76,E4,44.3,22.95,37.999
77,E5,54.48,32.93,51.797
78,E6,70.6,33.92,58.547
79,E1,84.63,27.78,60.936
80,E2,40.2,28.4,40.956
81,E3,49.86,20.18,38.17
82,E4,55.71,32.99,50.856
83,E5,81.07,20.25,51.618
84,E6,35.53,23.63,34.838
85,E1,79.43,34.2,62.906
86,E2,70.46,33.26,57.608
87,E3,26.77,19.17,26.075
88,E4,89.25,21.83,56.595
89,E5,46.35,30.81,45.15
90,E6,56.06,28.11,47.953
1,E1,22.37,34.47,38.407
"""
with open("engineering_data.csv", "w") as _f:          # your own copy of the data file: the same rows every time you run this cell
    _f.write(_CSV_TEXT)

def records_from_table(table):
    """A list of dictionaries, one per row. Missing values (NaN) become None."""
    objects = table.astype(object)
    return objects.where(table.notna(), None).to_dict("records")

raw_table = pd.read_csv("engineering_data.csv")
all_records = records_from_table(raw_table)
development_records = [r for r in all_records if r["equipment_id"] != "E6"]   # E6 is the TEST SET: nothing you write may use it before the very end
print("Rows in the file:", len(raw_table))
print("Rows you may develop with (groups E1 to E5):", len(development_records))
print(raw_table.head())


## Task 1: Clean the rows and look at them (2 automatic marks, Weeks 3 to 7)

A **record** is one row written as a dictionary: `{"record_id": 1, "equipment_id": "E1", "load_pct": 59.15, "ambient_c": 31.71, "target": 52.5}`. A missing value (a blank cell, `NaN` in pandas) is `None`.

`clean_rows(records)` returns a cleaned **copy** of the records, in the same order:
- keep only the first row for each `record_id`;
- drop a row whose `target` is `None`, or whose `load_pct` is `None` or outside 0 to 100 (0 and 100 are valid);
- **do not fill** `ambient_c`. A missing value stays `None`. The models do the imputation later, using the training set only, so that nothing leaks (data leakage).

Never change the list you were given. Then use your function on the development rows and draw **Chart 1**: `load_pct` across, `target` up, with a title and both axis names, and the unit (kW).

In [ ]:
def clean_rows(records):
    """Return a cleaned copy of the records (see the task text above)."""
    # TODO 1: make an empty list called kept and an empty set called seen.
    # TODO 2: loop over records. If record["record_id"] is already in seen, skip the row (continue). Otherwise add the id to seen.
    # TODO 3: skip the row if target is None, or load_pct is None, or load_pct is below 0 or above 100.
    # TODO 4: add dict(record) (a copy) to kept. Do not touch ambient_c.
    return None   # TODO 5: return kept instead


In [ ]:
development_rows = None      # TODO: call clean_rows on development_records
print("Rows kept:", None)    # TODO: replace None with the number of rows in development_rows

figure, axes = plt.subplots()
# TODO: axes.scatter(...): load_pct on the x axis, target on the y axis (hint: table = pd.DataFrame(development_rows))
# TODO: axes.set_title(...), axes.set_xlabel(...) and axes.set_ylabel(...). Write the unit in the y label.
chart_1 = figure
plt.show()


## Task 2: Split by whole equipment groups (2 automatic marks, Week 10)

`group_split(records)` returns a dictionary with three lists of `record_id` values, each in the order the records came in:
- `"train"`: groups E1, E2, E3 and E4 (the **training set**);
- `"validation"`: group E5 (the **validation set**);
- `"test"`: group E6 (the **test set**).

Never use the target to decide where a row goes. Whole machines go together, because a new machine must be a machine the model has never met. Your development rows have no E6 rows, so their `"test"` list is empty. That is correct.

In [ ]:
def group_split(records):
    """Return {"train": [...], "validation": [...], "test": [...]} (lists of record_id, in input order)."""
    # TODO: for each record, look at record["equipment_id"] and put record["record_id"] in the right list.
    return None


In [ ]:
split_ids = None     # TODO: group_split(development_rows)
train_rows = None    # TODO: the development rows whose record_id is in split_ids["train"]
valid_rows = None    # TODO: the development rows whose record_id is in split_ids["validation"]
print("Training rows:", None, "| validation rows:", None)   # TODO: replace the Nones with the two counts


## Task 3: Question 1: a baseline model, a linear regression and a decision tree (5 automatic marks, Weeks 8 and 9)

**Question 1.** How many kW of building cooling load will a building show at a given load and ambient temperature, for equipment the model has never met?

`fit_predict_regression(training_records, feature_records)` trains (fits) three models on the training records and predicts for the feature records. Use only `load_pct` then `ambient_c`, in that order. The feature records have **no target**.
- every model is a pipeline that first does **imputation** of missing values with the **median of the training rows** (`SimpleImputer(strategy="median")`), then fits;
- `"baseline"`: the baseline model, which always predicts the mean training target (`DummyRegressor`);
- `"linear"`: multiple linear regression (two features), `LinearRegression`;
- `"tree"`: a decision tree, `DecisionTreeRegressor(max_depth=4, random_state=17)`. The `max_depth` is a **hyperparameter**: a setting you choose before training.

Return a dictionary with three lists of predictions, in the order of the feature records. The baseline model must use the **training** rows only: a baseline built with validation or test targets has **leaked**.

`regression_errors(actual, predicted)` returns `{"mae": ..., "rmse": ...}` over the same pairs, not rounded. A **residual** is actual minus predicted. MAE (mean absolute error) is the mean of the absolute residuals. RMSE (root mean squared error) is the square root of the mean of the squared residuals.

In [ ]:
def fit_predict_regression(training_records, feature_records):
    """Return {"baseline": [...], "linear": [...], "tree": [...]}."""
    # TODO 1: table = pd.DataFrame(training_records); new = pd.DataFrame(feature_records)
    # TODO 2: X_train = the two feature columns of table; y_train = table["target"]; X_new = the same two columns of new.
    # TODO 3: a dictionary of three pipelines. Each is make_pipeline(SimpleImputer(strategy="median"), <the model>).
    # TODO 4: loop over the dictionary: fit on (X_train, y_train), predict X_new, and store predictions.tolist() under the model's name.
    return None


In [ ]:
def regression_errors(actual, predicted):
    """Return {"mae": ..., "rmse": ...}, not rounded."""
    # TODO: use numpy: residuals = np.array(actual) - np.array(predicted); MAE uses np.abs and the mean; RMSE uses squares, the mean, then np.sqrt.
    return None


In [ ]:
# Evaluate the three models on the validation set (group E5). Keep the names: validation_predictions, validation_mae and validation_rmse.
validation_features = None    # TODO: the validation rows without the "target" key
validation_actual = None      # TODO: the list of validation targets
validation_predictions = None # TODO: fit_predict_regression(train_rows, validation_features)
validation_mae = None         # TODO: a dictionary {"baseline": ..., "linear": ..., "tree": ...} of MAE values, from regression_errors
validation_rmse = None        # TODO: the same with RMSE
print("Validation MAE  (" + TARGET_UNIT + "):", validation_mae)
print("Validation RMSE (" + TARGET_UNIT + "):", validation_rmse)

figure, axes = plt.subplots()
# TODO: a bar chart of the three MAE values, with a title, a y label that includes the unit, and the model names on the x axis.
chart_2 = figure
plt.show()


## Task 4: Question 2: which rows should raise a warning? (5 automatic marks, Week 9)

**Question 2.** A row is a **warning** (1, the positive class) when its target is at or above `LIMIT` (54 kW) and **no warning** (0, the negative class) below it. Which rows should raise a warning?

`warning_probabilities(training_records, feature_records, limit)` returns the probability of a warning, for each feature record, from a pipeline of `SimpleImputer(strategy="median")`, `StandardScaler()` and `LogisticRegression(max_iter=1000)` (binary classification), fitted on the training records only. The label is `1` when the training target is `>= limit`. Use `load_pct` then `ambient_c`.

`classification_scores(actual, probability, threshold)` predicts a warning when the probability is `>= threshold` (the **decision threshold**), then returns a dictionary with the confusion matrix counts `tp`, `fp`, `fn`, `tn` (true positives, false positives, false negatives, true negatives) and `precision`, `recall`, `f1`, `cost` and `auc`. A false negative (a missed warning) costs `MISSED_COST` and a false positive (a false alarm) costs `FALSE_ALARM_COST` (invented teaching costs), so `cost = 5 * fn + 1 * fp`. When a ratio has a zero denominator, use 0. `auc` is the ROC AUC of the probabilities and does not depend on the threshold.

Then try decision thresholds 0.3, 0.5 and 0.7 on the validation set, and keep the one with the lowest cost as `chosen_threshold` (if two tie, take the higher one).

In [ ]:
def warning_probabilities(training_records, feature_records, limit):
    """Return a list of warning probabilities, one per feature record."""
    # TODO 1: tables as before; y_train = (training target >= limit) turned into 0 and 1 (astype(int)).
    # TODO 2: model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)); fit it.
    # TODO 3: predict_proba(X_new) gives two columns: column 1 is the probability of a warning. Return it as a list of floats.
    return None


In [ ]:
def classification_scores(actual, probability, threshold):
    """Return tp, fp, fn, tn, precision, recall, f1, cost and auc."""
    # TODO 1: predicted = 1 where the probability is at or above the threshold, else 0 (a list comprehension).
    # TODO 2: tn, fp, fn, tp = confusion_matrix(actual, predicted).ravel()   (see Class 2)
    # TODO 3: precision_score, recall_score and f1_score with zero_division=0; roc_auc_score(actual, probability).
    # TODO 4: cost = MISSED_COST * fn + FALSE_ALARM_COST * fp. Turn numpy numbers into int() and float() before returning.
    return None


In [ ]:
warning_actual = None        # TODO: 1 or 0 for each validation row: is the target at or above LIMIT?
warning_probability = None   # TODO: warning_probabilities(train_rows, validation_features, LIMIT)
threshold_table = {}         # TODO: for each threshold in 0.3, 0.5, 0.7: threshold_table[threshold] = classification_scores(...)
warning_auc = None           # TODO: the "auc" value of any one entry of threshold_table
chosen_threshold = None      # TODO: the threshold with the lowest cost (if two tie, the higher threshold)
for threshold, scores in threshold_table.items():
    print(threshold, scores)
print("Chosen threshold:", chosen_threshold)


## Task 5: Can you trust it? Group cross-validation (3 automatic marks, Week 10)

One validation set of about 15 rows can mislead. **Group cross-validation** (K-fold cross-validation where each fold is a whole equipment group, the idea of `GroupKFold`) gives each machine a turn as the hidden machine.

`group_cv_mae(records)` receives clean records with **no missing values** and uses whole equipment groups. Hold out each of E1, E2, E3, E4 and E5 in turn (E6 is never used). For each fold, fit `LinearRegression` on `load_pct` and `ambient_c` using the other four groups, and compute the MAE on the held-out group. Return `{"folds": [...], "mean": ..., "spread": ...}`: the five MAEs in the order E1 to E5, their mean, and their sample standard deviation (`ddof=1`). Do not round.

**Data leakage check, in your own head:** the imputer, the StandardScaler (standardisation) and the baseline model's mean are all fitted on the training rows only. If any of them had seen validation or test rows, the scores would look better than they are.

In [ ]:
def group_cv_mae(records):
    """Return {"folds": [five MAEs, E1 to E5], "mean": ..., "spread": ...}."""
    # TODO 1: table = pd.DataFrame(records)
    # TODO 2: for group in ["E1", "E2", "E3", "E4", "E5"]: train = rows of the OTHER groups in E1 to E5, test = the rows of group.
    # TODO 3: fit LinearRegression on the two feature columns of train; the MAE on test is np.mean(np.abs(...)).
    # TODO 4: mean = np.mean(folds); spread = np.std(folds, ddof=1).
    return None


In [ ]:
complete_rows = None    # TODO: the development rows whose ambient_c is not None
cv_summary = None       # TODO: group_cv_mae(complete_rows)
print("Cross-validated MAE (" + TARGET_UNIT + "):", cv_summary)


## Task 6: Does a new column help? An ablation study (2 automatic marks, Week 11)

`ablation_mae(training_records, validation_records)` is a small **ablation study**: it compares two linear regression models on the same rows. Both records lists are complete (no missing values). Fit `LinearRegression` on the training records twice: with the two base features (`load_pct`, `ambient_c`), and with a third column, the **interaction feature** `load_pct * ambient_c`. Score both on the validation records. Return `{"base_mae": ..., "extended_mae": ..., "improvement": base_mae - extended_mae}`, not rounded. A positive improvement means the new feature helped.

In [ ]:
def ablation_mae(training_records, validation_records):
    """Return base_mae, extended_mae and improvement."""
    # TODO 1: two tables. Build the extra column on both: table["load_x_ambient"] = table["load_pct"] * table["ambient_c"].
    # TODO 2: fit LinearRegression with the base columns, then with the base columns plus load_x_ambient. Same rows, same measure.
    return None


In [ ]:
complete_train = None    # TODO: the rows of train_rows whose ambient_c is not None
complete_valid = None    # TODO: the rows of valid_rows whose ambient_c is not None
ablation_result = None   # TODO: ablation_mae(complete_train, complete_valid)
print("Ablation (" + TARGET_UNIT + "):", ablation_result)


## Task 7: Choose one model by a rule (1 automatic marks, Week 11)

`choose_model(candidates, max_mae, max_ms)` takes a list of dictionaries with `name`, `validation_mae` and `latency_ms`. Keep the candidates whose `validation_mae <= max_mae` **and** `latency_ms <= max_ms` (both limits are inclusive). Among those, choose the smallest `validation_mae`; break a tie by the smaller `latency_ms`, then by the name in alphabetical order. Return the chosen name, or `None` if nobody qualifies. Never use the test set to choose.

The client for this project is the client who pays for the cooling plant. The client accepts a typical prediction error of at most `MAX_MAE` (2 kW) and a prediction time of at most `MAX_MS` (50 ms). Both numbers are invented for teaching.

In [ ]:
def choose_model(candidates, max_mae, max_ms):
    """Return the chosen model name, or None."""
    # TODO: filter with a list comprehension, then sort with key=lambda c: (c["validation_mae"], c["latency_ms"], c["name"]).
    return None


In [ ]:
candidate_table = []     # TODO: one dictionary per model (baseline, linear, tree): name, validation_mae (from your Task 3 results), latency_ms (from LATENCY_MS)
chosen_model = None      # TODO: choose_model(candidate_table, MAX_MAE, MAX_MS)
print(candidate_table)
print("Chosen model:", chosen_model)


## Task 8: Open the test set, once (4 automatic marks, Week 11)

Group E6 is the **test set**: a different piece of equipment that nothing so far has used. It gives **one** honest estimate for the model you chose.

How it works. The supplied cell below is marked Do not edit this cell. Run it once to define `open_test_set(model_name, threshold)`. Then call it **one time**, with the model you chose in Task 7 and the decision threshold you chose in Task 4. It trains your functions on all the development rows (E1 to E5), evaluates on E6, prints the result and writes your choice to a small file. If you call it again, the notebook counts a second opening: you lose the "opened once" mark even when the choice is the same, and you lose more if the choice is different.

You must not use E6 to choose a model, a threshold or anything else, and you must not look at it before this call. Report whatever the result is in your Part B form.

In [ ]:
#@title Open the test set (run it ONCE, at the very end) - Do not edit this cell
# IUEA-TEST-CELL  (supplied: do not edit)
import json as _json
TEST_OPEN_LOG = []
_TEST_FILE = "test_first_opening.json"

def open_test_set(model_name, threshold):
    """Evaluate ONE chosen model and ONE chosen decision threshold on the test set (group E6), with the functions you wrote. Returns a dictionary."""
    if model_name not in ("baseline", "linear", "tree"):
        raise ValueError('model_name must be "baseline", "linear" or "tree"')
    if not isinstance(threshold, (int, float)) or not 0 < threshold < 1:
        raise ValueError("threshold must be a number between 0 and 1")
    first = _json.load(open(_TEST_FILE)) if os.path.exists(_TEST_FILE) else None
    if first is None:
        _json.dump({"model": model_name, "threshold": threshold}, open(_TEST_FILE, "w"))
    TEST_OPEN_LOG.append([model_name, threshold])
    changed = first is not None and (first["model"] != model_name or first["threshold"] != threshold)
    if changed:
        print("WARNING: THE TEST SET WAS ALREADY OPENED with model", first["model"], "and threshold", first["threshold"],
              "- opening it again with a different choice means it is no longer an honest test.")
    table = pd.read_csv("engineering_data.csv").drop_duplicates("record_id")
    table = table[table["target"].notna() & table["load_pct"].between(0, 100)]
    development = records_from_table(table[table["equipment_id"] != "E6"])
    held_out = records_from_table(table[table["equipment_id"] == "E6"])
    features = [{k: v for k, v in r.items() if k != "target"} for r in held_out]
    actual = [r["target"] for r in held_out]
    errors = regression_errors(actual, fit_predict_regression(development, features)[model_name])
    probability = warning_probabilities(development, features, LIMIT)
    warning = classification_scores([1 if v >= LIMIT else 0 for v in actual], probability, threshold)
    outcome = {"model": model_name, "threshold": threshold, "n_rows": len(held_out), "mae": errors["mae"], "rmse": errors["rmse"],
               "warning": warning, "changed_after_opening": bool(changed)}
    print("Test set opened with", model_name, "and threshold", threshold, "on", len(held_out), "rows")
    print("  MAE", round(errors["mae"], 3), TARGET_UNIT, "| RMSE", round(errors["rmse"], 3), TARGET_UNIT)
    print("  warnings: precision", round(warning["precision"], 3), "recall", round(warning["recall"], 3), "F1", round(warning["f1"], 3), "AUC", round(warning["auc"], 3))
    return outcome


In [ ]:
test_result = None     # TODO: open_test_set(chosen_model, chosen_threshold)


## Self-check (not marked)
Run this supplied cell to see which functions still need work. It uses tiny examples with known answers. The marked checks use other data.

In [ ]:
# This cell is a SELF-CHECK. It is not marked. It uses tiny examples with known answers, so you can see at once if a function is wrong.
# The marked (hidden) checks run your functions on OTHER data you cannot see, so passing here does not mean full marks.
def _try(label, test):
    try:
        ok = bool(test())
    except Exception as error:
        ok = False
    print(("PASS  " if ok else "TODO  ") + label)
    return ok

_small = [dict(record_id=1, equipment_id="E1", load_pct=50.0, ambient_c=20.0, target=10.0),
          dict(record_id=1, equipment_id="E1", load_pct=50.0, ambient_c=20.0, target=10.0),
          dict(record_id=2, equipment_id="E2", load_pct=120.0, ambient_c=22.0, target=12.0),
          dict(record_id=3, equipment_id="E5", load_pct=40.0, ambient_c=None, target=None),
          dict(record_id=4, equipment_id="E6", load_pct=100.0, ambient_c=None, target=20.0)]
_checks = []
_checks.append(_try("clean_rows keeps ids 1 and 4 only, in order, and leaves the missing ambient_c as None",
                    lambda: [r["record_id"] for r in clean_rows(_small)] == [1, 4] and clean_rows(_small)[1]["ambient_c"] is None))
_checks.append(_try("clean_rows does not change the list you give it", lambda: (len(_small) == 5 and clean_rows(_small) is not _small)))
_checks.append(_try("group_split puts E1 in train, E5 in validation, E6 in test",
                    lambda: group_split(_small) == {"train": [1, 1, 2], "validation": [3], "test": [4]}))
_toy = [dict(record_id=i, equipment_id="E1", load_pct=10.0 * i, ambient_c=20.0, target=2.0 * i) for i in range(1, 7)]
_new = [dict(record_id=9, equipment_id="E2", load_pct=35.0, ambient_c=None)]
_checks.append(_try("fit_predict_regression returns baseline, linear and tree, one prediction per feature record",
                    lambda: sorted(fit_predict_regression(_toy, _new)) == ["baseline", "linear", "tree"] and all(len(v) == 1 for v in fit_predict_regression(_toy, _new).values())))
_checks.append(_try("the baseline model predicts the mean of the TRAINING targets (7.0)", lambda: abs(fit_predict_regression(_toy, _new)["baseline"][0] - 7.0) < 1e-9))
_checks.append(_try("regression_errors([1, 3], [2, 5]) has mae 1.5 and rmse sqrt(2.5)",
                    lambda: abs(regression_errors([1, 3], [2, 5])["mae"] - 1.5) < 1e-9 and abs(regression_errors([1, 3], [2, 5])["rmse"] - 2.5 ** 0.5) < 1e-9))
_checks.append(_try("classification_scores on a known case (tp 1, fp 1, fn 1, tn 1, cost 6, auc 0.75)",
                    lambda: (lambda s: (s["tp"], s["fp"], s["fn"], s["tn"], s["cost"], round(s["auc"], 2)) == (1, 1, 1, 1, 6, 0.75))(classification_scores([1, 0, 1, 0], [0.9, 0.8, 0.2, 0.1], 0.5))))
_checks.append(_try("warning_probabilities gives numbers between 0 and 1",
                    lambda: all(0 <= p <= 1 for p in warning_probabilities(_toy, _new, 8.0))))
_checks.append(_try("choose_model picks the smallest error that meets both limits",
                    lambda: choose_model([dict(name="a", validation_mae=2, latency_ms=3), dict(name="b", validation_mae=1, latency_ms=30)], 2, 10) == "a"))
_checks.append(_try("group_cv_mae returns five folds, a mean and a spread",
                    lambda: len(group_cv_mae(complete_rows)["folds"]) == 5 and {"mean", "spread"} <= set(group_cv_mae(complete_rows))))
_checks.append(_try("ablation_mae returns base_mae, extended_mae and improvement",
                    lambda: {"base_mae", "extended_mae", "improvement"} <= set(ablation_mae(complete_train, complete_valid))))
for _name in ["validation_mae", "cv_summary", "warning_auc", "ablation_result", "candidate_table", "chosen_model", "chosen_threshold"]:
    _checks.append(_try("the variable " + _name + " has been calculated (it is not None)", lambda: globals().get(_name) is not None))
_checks.append(_try("the test set has been opened exactly once", lambda: len(TEST_OPEN_LOG) == 1))
_checks.append(_try("Chart 1 and Chart 2 each have a title, named axes and the unit",
                    lambda: all(f.axes[0].get_title() and f.axes[0].get_xlabel() and f.axes[0].get_ylabel() for f in (chart_1, chart_2))))
print()
print(sum(_checks), "of", len(_checks), "self-checks pass. Now choose Runtime, Restart session and run all, and check that nothing stops with an error.")


## Task 9: Part B, the decision memo form (15 automatic marks)
Your memo is a **form**, not an essay. You write it for **the client who pays for the cooling plant**. Fill in the dictionary `MEMO` in the next cell. Every value is checked by the marking computer; nothing is read by a person. Use your own numbers, with at least two decimals, in **kW**.

| Field | What to enter | Marks |
|---|---|---|
| `recommended_model`, `recommended_threshold` | One of the models you trained (`"baseline"`, `"linear"`, `"tree"`) and one decision threshold (0.3, 0.5 or 0.7). They must be the model and threshold your own rule and cost table chose. | 3 |
| `validation_mae` | The validation MAE of each of the three models, as your notebook computed them. | 3 |
| `test_mae`, `test_recall` | The MAE and the warning recall on the test set from your ONE opening (recall is between 0 and 1). The marking computer recomputes them from your own functions. | 2 |
| `cv_mean_mae` | The mean MAE from your group cross-validation. | 1 |
| `limits` | The letters of the **three** statements below that are **true limits of this study**. Three different letters. One mark for each true one. | 3 |
| `next_step` | One letter: the best next step. | 1 |
| `answer_choice`, `answer_test_set` | Two short written answers of 8 to 80 words each. One mark each. | 2 |

**Limits.** Choose the three statements that are TRUE and are limits of this study (some statements are false):
- **A.** The validation set is one equipment group with only about 15 rows, so the validation errors could change a lot with a different group.
- **B.** The data are synthetic (invented for teaching), so the numbers say nothing certain about real buildings.
- **C.** The client limits, the decision threshold options and the costs (5 and 1) are invented teaching numbers; a real client would set their own.
- **D.** Because the test set was opened only once, the test metrics have no uncertainty at all.
- **E.** The model with the lowest training error is always the best model for new equipment.
- **F.** Fitting the imputer on the training set only makes the validation results unreliable.
- **G.** The test MAE proves that the model is safe on every building the client owns.
- **H.** RMSE is always smaller than MAE, so MAE is the more severe measure and should be the only one reported.
- **I.** The test set is a single equipment group, so the test metrics are one estimate that may be luckier or unluckier than the truth.
- **J.** The baseline model should be left out of the comparison because it uses no features.

**Next step.** Choose the one best next step:
- **A.** Collect real measurements from more buildings and validate the chosen model again with group cross-validation before any decision.
- **B.** Move the test set into the training set so the model learns from more rows, and report the training error.
- **C.** Deploy the chosen model now, because the test MAE is below the client limit.
- **D.** Remove the baseline model from the report so the results look better.
- **E.** Tune the maximum depth of the decision tree until its test MAE is lowest, then report that test MAE.

**Written answers.** Write in your own words, in full sentences.
- `answer_choice`: In one or two sentences: why do you recommend this model? Use a number from the validation set and say what you compared it with.
- `answer_test_set`: In one or two sentences: why is the test set opened only once, at the very end, and never used to choose the model?

The lettering of these lists is different for each student, so a list of letters from a classmate will not match your lists.

In [ ]:
#@title Part B: the decision memo form (fill in every field; do not rename a field)
MEMO = {
    "recommended_model": "",                  # "baseline", "linear" or "tree"
    "recommended_threshold": None,            # 0.3, 0.5 or 0.7
    "validation_mae": {"baseline": None, "linear": None, "tree": None},   # your validation MAE values, in kW
    "cv_mean_mae": None,                      # your mean cross-validated MAE, in kW
    "test_mae": None,                         # the test MAE from your ONE opening, in kW
    "test_recall": None,                      # the test recall from your ONE opening (0 to 1)
    "limits": ["", "", ""],                   # three different letters from the Limits list
    "next_step": "",                          # one letter from the Next step list
    "answer_choice": "",                      # one or two sentences
    "answer_test_set": "",                    # one or two sentences
}


## Memo self-check (not marked)
Run this supplied cell after you fill in the form. It tells you which fields are empty and whether your numbers agree with your own notebook.

In [ ]:
# MEMO SELF-CHECK (not marked): did you fill in every field, and do your numbers agree with your own notebook?
# It does not tell you whether your choices are the best ones. The marking computer checks that.
def _memo_try(label, test):
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("PASS  " if ok else "TODO  ") + label)
    return ok

def _near(a, b):
    return isinstance(a, (int, float)) and isinstance(b, (int, float)) and abs(a - b) <= 0.011

_memo_checks = [
    _memo_try("recommended_model is one of the models you trained", lambda: MEMO["recommended_model"] in validation_mae),
    _memo_try("recommended_threshold is 0.3, 0.5 or 0.7", lambda: MEMO["recommended_threshold"] in (0.3, 0.5, 0.7)),
    _memo_try("validation_mae has your three validation MAE values (two decimals are enough)", lambda: all(_near(MEMO["validation_mae"][k], validation_mae[k]) for k in ("baseline", "linear", "tree"))),
    _memo_try("cv_mean_mae is your cross-validated mean MAE", lambda: _near(MEMO["cv_mean_mae"], cv_summary["mean"])),
    _memo_try("test_mae is the test MAE you saw when you opened the test set", lambda: _near(MEMO["test_mae"], test_result["mae"])),
    _memo_try("test_recall is the test recall you saw (between 0 and 1)", lambda: _near(MEMO["test_recall"], test_result["warning"]["recall"])),
    _memo_try("limits has three different letters", lambda: len({str(x).strip().upper() for x in MEMO["limits"] if str(x).strip()}) == 3),
    _memo_try("next_step is one letter", lambda: len(str(MEMO["next_step"]).strip()) == 1),
    _memo_try("answer_choice has 8 to 80 words", lambda: 8 <= len(str(MEMO["answer_choice"]).split()) <= 80),
    _memo_try("answer_test_set has 8 to 80 words", lambda: 8 <= len(str(MEMO["answer_test_set"]).split()) <= 80),
]
print()
print(sum(_memo_checks), "of", len(_memo_checks), "memo self-checks pass.")


## Save and submit (allow at least 20 minutes)
1. Choose **Runtime, then Restart session and run all**. Every cell must run without an error. Fix the first error first.
2. Check your identity cell, your two charts and your `MEMO` form. Choose **File, then Save**.
3. **Run, do not edit:** the **Submit your test** cell below. Choose the same Google account when asked. Click **Submit test** and wait for the form link.
4. In the form, check the account email, your name and registration number. Leave the prefilled snapshot fields unchanged. Press **Submit** before your personal deadline.
5. Wait for **Your response has been recorded**. Keep the saved snapshot in Drive until marks are released.

A saved notebook, a snapshot or an open form is not a submission: **you must submit the form**. Submit your work even if some tasks are unfinished. A new submission before the deadline must be a complete notebook; the latest one on time is marked. When the time ends the form no longer counts, Colab may still let you type but nothing after the deadline is marked.

**Declaration:** by submitting, I confirm that I completed this project myself without AI assistance or another person's help, using only the permitted resources. I can explain and change the code I submitted.

In [ ]:
#@title Submit your test — Do not edit this cell
# Do not edit this cell: saves your answers and prepares the submission form.
# Submission helper only. This cell is excluded from marking.
# Run this cell, then click Submit test. Never share credentials or tokens.
import copy as _copy
import hashlib as _hashlib
import io as _io
import json as _json
import urllib.parse as _urlparse
import uuid as _uuid
import datetime as _datetime
import time as _time
import threading as _threading
import ipywidgets as _widgets
from IPython.display import display as _display, HTML as _HTML, clear_output as _clear_output

_EXAM_CONFIG = {'lecturer_email': 'wirmbeesubs@gmail.com', 'forms': {}}
_EXPIRED_MESSAGE = 'Time has ended. Submission is closed. Do not use an old form link. Contact your lecturer if you need help.'

class _IUEAAccountMismatch(RuntimeError):
    def __init__(self, active_email, allocated_email, snapshot_owner=''):
        self.active_email = active_email or 'could not be verified'
        self.allocated_email = allocated_email or 'could not be verified'
        self.snapshot_owner = snapshot_owner

def _show_account_warning(error):
    import html as _html
    active = _html.escape(error.active_email)
    allocated = _html.escape(error.allocated_email)
    owner = ''
    if error.snapshot_owner:
        owner = '<p><strong>Snapshot owner:</strong> '+_html.escape(error.snapshot_owner)+'</p>'
    _display(_HTML(
        '<div role="alert" style="border:4px solid #b3261e;background:#fff0ee;'
        'padding:18px;margin:12px 0;color:#7a1712;font-size:17px">'
        '<p style="font-size:21px;font-weight:bold;margin-top:0">'
        'ACCOUNT MISMATCH — submission link withheld</p>'
        '<p><strong>Currently active Colab account:</strong> '+active+'</p>'
        '<p><strong>Account allocated to this test:</strong> '+allocated+'</p>'
        +owner+
        '<p>Switch to the allocated Google account inside Colab, then rerun this '
        'submission cell. Do not submit while these accounts differ.</p></div>'
    ))

def _submission_expired():
    _deadline = globals().get('_IUEA_DEADLINE', '')
    if not _deadline:
        if globals().get('_IUEA_ALLOCATION') and TEST_ID in ('TEST1', 'ML_PROJECT'):
            raise RuntimeError('The saved deadline is missing. Reopen your issued notebook through the test entry page; do not start another attempt.')
        return False
    return _time.time() >= _datetime.datetime.fromisoformat(_deadline.replace('Z', '+00:00')).timestamp()

def _check_submission_time():
    if _submission_expired():
        raise RuntimeError(_EXPIRED_MESSAGE)

# Check before requesting Google access or creating any snapshot.
_check_submission_time()
# Authenticate while running the cell so Colab can display its account dialog.
from google.colab import auth as _colab_auth
print('Double-check your full name, Student ID/ Registration Number, and email before submitting.')
print('Name:', FULL_NAME, '| Student ID/ Registration Number:', STUDENT_ID, '| Email:', EMAIL)
print('Prepare submission: authorize your own Google account if asked.')
_colab_auth.authenticate_user()

_button = _widgets.Button(description="Submit test", button_style="success", icon="check")
_submission_output = _widgets.Output()

def _prepare_test_submission(_):
    _button.disabled = True
    with _submission_output:
        _clear_output()
        try:
            _check_submission_time()
            for _label,_value in [('Full name',FULL_NAME),('Student ID/ Registration Number',STUDENT_ID),('Email',EMAIL)]:
                if not isinstance(_value,str) or not _value.strip():
                    raise ValueError('Complete '+_label+' in the identity cell and run that cell first.')
            _key = COHORT+'|'+TEST_ID
            _form = _EXAM_CONFIG['forms'][_key]
            if VARIANT not in ('A','B','C'):raise ValueError('Keep your assigned variant A, B or C.')
            from google.colab import auth as _auth, _message
            import google.auth as _google_auth
            from googleapiclient.discovery import build as _build
            from googleapiclient.http import MediaIoBaseUpload as _MediaUpload
            print('Preparing your notebook snapshot. Use the same Google account in the final form.')
            import logging as _logging
            _logging.getLogger('google_auth_httplib2').addFilter(lambda r: not r.getMessage().startswith('httplib2 transport does not support per-request timeout'))
            _credentials,_project = _google_auth.default()
            _drive = _build('drive','v3',credentials=_credentials,cache_discovery=False)
            _reply = _message.blocking_request('get_ipynb',timeout_sec=60)
            _notebook = _copy.deepcopy(_reply['ipynb'])
            _metadata = _notebook.get('metadata',{})
            _allocated_email = str(
                _metadata.get('iuea_allocated_email')
                or globals().get('_IUEA_ALLOCATED_EMAIL')
                or EMAIL
            ).strip().lower()
            _metadata_code = str(_metadata.get('iuea_allocation_code') or '')
            _runtime_code = str(globals().get('_IUEA_ALLOCATION') or '')
            if not _allocated_email:
                raise ValueError('The issued notebook has no allocated Google account. Return to the test entry page and contact your lecturer.')
            if _metadata_code and _runtime_code and _metadata_code != _runtime_code:
                raise ValueError('This notebook allocation does not match the running identity cell. Reopen your issued notebook from the test entry page.')
            _active_email = str(
                (_drive.about().get(fields='user(emailAddress)').execute()
                 .get('user',{}).get('emailAddress',''))
            ).strip().lower()
            if not _active_email or _active_email != _allocated_email:
                raise _IUEAAccountMismatch(_active_email, _allocated_email)
            if str(EMAIL).strip().lower() != _allocated_email:
                raise ValueError('The notebook identity email does not match the allocation. Reopen your issued notebook from the test entry page.')
            # Remove outputs from the submission helper, which contain temporary links.
            for _cell in _notebook['cells']:
                if 'iuea-submission' in _cell.get('metadata',{}).get('tags',[]):
                    _cell['outputs']=[];_cell['execution_count']=None
            _payload = _json.dumps(_notebook,ensure_ascii=False).encode('utf-8')
            if len(_payload)>10*1024*1024:raise ValueError('Notebook exceeds 10 MB. Clear large outputs, then try again.')
            _sha = _hashlib.sha256(_payload).hexdigest()
            _name = COHORT+'_'+TEST_ID+'_submission_'+_uuid.uuid4().hex[:12]+'.ipynb'
            _check_submission_time()
            _created = _drive.files().create(body={'name':_name,'mimeType':'application/x-ipynb+json'},media_body=_MediaUpload(_io.BytesIO(_payload),mimetype='application/x-ipynb+json',resumable=False),fields='id').execute()
            _snapshot = _drive.files().get(
                fileId=_created['id'],fields='id,owners(emailAddress)'
            ).execute()
            _owners = _snapshot.get('owners',[])
            _snapshot_owner = str(
                _owners[0].get('emailAddress','') if _owners else ''
            ).strip().lower()
            if not _snapshot_owner or _snapshot_owner != _allocated_email:
                raise _IUEAAccountMismatch(
                    _active_email,_allocated_email,
                    _snapshot_owner or 'could not be verified'
                )
            _drive.permissions().create(fileId=_created['id'],body={'type':'user','role':'reader','emailAddress':_EXAM_CONFIG['lecturer_email']},sendNotificationEmail=False).execute()
            _values={'Full name':FULL_NAME.strip(),'Student Registration Number':STUDENT_ID.strip(),'Variant':globals().get('_IUEA_ALLOCATION', VARIANT),'Snapshot file ID':_created['id'],'Snapshot SHA-256':_sha}
            _parts = _urlparse.urlsplit(_form['prefill'])
            _query = _urlparse.parse_qsl(_parts.query,keep_blank_values=True)
            _replace={'PLACEHOLDER_'+str(_form['items'][k]):str(v) for k,v in _values.items()}
            _query=[(k,_replace.get(v,v)) for k,v in _query]
            # Select the student's account when several Google accounts are signed in.
            _query=[(k,v) for k,v in _query if k.lower() != 'authuser']
            _query.append(('authuser', EMAIL.strip().lower()))
            _link = _urlparse.urlunsplit((_parts.scheme,_parts.netloc,_parts.path,_urlparse.urlencode(_query),_parts.fragment))
            import html as _html
            _check_submission_time()
            _display(_HTML('<a target="_blank" rel="noopener" href="'+_html.escape(_link,quote=True)+'" style="font-size:18px">Finish submission: open the form and press Submit</a>'))
            print('Snapshot prepared. You have NOT submitted yet. Open the link above, confirm your details, tick the email confirmation box and press Submit before the deadline. Keep the confirmation page. Do not edit the snapshot file in Drive.')
        except _IUEAAccountMismatch as _error:
            _show_account_warning(_error)
            print('No submission link was created. Switch to the allocated account and rerun this cell.')
        except Exception as _error:
            print('Submission was not completed:',str(_error)[:300])
            print(_EXPIRED_MESSAGE if _submission_expired() else 'If time remains, retry or contact your lecturer before the deadline. Do not assume your work was submitted.')
        finally:
            _button.disabled = _submission_expired()

_button.on_click(_prepare_test_submission)
_display(_button,_submission_output)

# Disable the displayed button when the fixed deadline arrives, even without a click.
if globals().get('_submission_expiry_timer'):
    _submission_expiry_timer.cancel()
if globals().get('_IUEA_DEADLINE'):
    def _close_submission_button():
        _button.disabled = True
        _button.description = 'Time has ended'
        with _submission_output:
            _clear_output()
            print(_EXPIRED_MESSAGE)
    _seconds_left = max(0, _datetime.datetime.fromisoformat(_IUEA_DEADLINE.replace('Z', '+00:00')).timestamp() - _time.time())
    _submission_expiry_timer = _threading.Timer(_seconds_left, _close_submission_button)
    _submission_expiry_timer.daemon = True
    _submission_expiry_timer.start()
